In [1]:
from dotenv import load_dotenv

load_dotenv()


True

# 2.4 Wedding Planner
In this lab you will build a multi-agent wedding planner.

> Note: This lab has been updated since filming to make the agent performance more robust to errors and to limit run time. In particular: 1) added error handling for MCP failures, 2) limited the number of searches. It's worth noting that, where possible, tool errors should be returned to the agent rather than failing so that the agent can adjust its invocation and try again.

## Setup Tools


In [2]:
import asyncio

from langchain_mcp_adapters.client import MultiServerMCPClient
from mcp.shared.exceptions import McpError
from mcp.types import CallToolResult, TextContent

RETRYABLE_MCP_CODES = {-32603}

class RetryMCPInterceptor:
    """Intercept MCP tool calls: retry transient failures, surface all errors gracefully.

    - Retryable McpError codes (e.g. -32603): retry with exponential backoff.
    - Non-retryable McpError codes (e.g. -32602): return error message immediately.
    - Any other exception (fetch failed, network errors, etc.): retry then return error message.
    """

    def __init__(self, max_retries: int = 3):
        self.max_retries = max_retries

    async def __call__(self, request, handler):
        last_error = None
        for attempt in range(self.max_retries):
            try:
                return await handler(request)
            except McpError as exc:
                last_error = exc
                print(f"[MCP interceptor] {type(exc).__name__} on {request.name} "
                      f"(code {exc.error.code}, attempt {attempt+1}/{self.max_retries}): {exc}")
                if exc.error.code not in RETRYABLE_MCP_CODES:
                    return CallToolResult(
                        content=[TextContent(type="text", text=f"Tool call failed (non-retryable): {exc}")],
                        isError=False,
                    )
            except Exception as exc:
                last_error = exc
                print(f"[MCP interceptor] {type(exc).__name__} on {request.name} "
                      f"(attempt {attempt+1}/{self.max_retries}): {exc}")

            if attempt < self.max_retries - 1:
                await asyncio.sleep(2 ** attempt)

        print(f"[MCP interceptor] all {self.max_retries} retries exhausted for {request.name}")
        return CallToolResult(
            content=[TextContent(type="text", text=f"Tool call failed after {self.max_retries} attempts: {last_error}")],
            isError=False,
        )

client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    },
    tool_interceptors=[RetryMCPInterceptor()],
)

tools = await client.get_tools()

In [22]:
from typing import Dict, Any
from tavily import TavilyClient
from langchain.tools import tool
from datetime import datetime

tavily_client = TavilyClient()

@tool
def web_search(query: str, search_number: int, max_search_number: int) -> Dict[str, Any]:
    """Search the web for information. You must track your search count by providing
    search_number (starting at 1) and max_search_number on every call.
    Queries must use only plain text characters. Do not use accented or special characters     
      (e.g., use 'capacite' instead of 'capacité').
    """
    if search_number > max_search_number:
        return {"message": "Search limit reached. Please summarize your findings and provide your final answer."}
    try:
        return tavily_client.search(query)
    except Exception as e:
        return {"error": str(e)}

@tool
def get_current_date() -> str:
    """Returns today's current date and year. 
    """
    return datetime.now().strftime("%Y-%m-%d")

In [4]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

@tool
def query_playlist_db(query: str) -> str:

    """Query the database for playlist information"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error querying database: {e}"

C:\Users\Sahal\AppData\Local\Temp\ipykernel_23424\2549610831.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


## Create State

In [5]:
from langchain.agents import AgentState

class WeddingState(AgentState):
    origin: str
    destination: str
    guest_count: str
    genre: str

## Create Subagents


In [30]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="gemma4",
    model_provider="openai",
    api_key="dummy",
    base_url="http://localhost:8080/v1"
)

# model = init_chat_model(
#     model="gemini-3.1-flash-lite",
#     model_provider="google-genai",
# )

# Travel agent
travel_agent = create_agent(
    model=model,
    tools=tools + [get_current_date],
    system_prompt="""
    You are a travel agent. Search for flights to the desired destination wedding location.
    RULES:
    1. NEVER ask the user questions or wait for confirmation.
    2. Based on today's date, choose the best future date within a year for visiting the destination (explain why in your response)
    and immediately execute the flight search tool with that date.
    3. Find 1 adult, one-way economy flights.
    4. Shortlist the best options based on lowest price and shortest duration.
    You may need to make multiple searches to iteratively find the best options.
    You will be given no extra information, only the origin and destination. It is your job to think critically about the best options.
    If the MCP tool fails, returns malformed output, or does not give you usable flight results, try the tool again.
    Once you have found the best options, let the user know your shortlist of options.
    """
)

In [7]:
# Venue agent
venue_agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt="""
    You are a venue specialist. Search for venues in the desired location, and with the desired capacity.
    **CRITICAL** You are not allowed to ask any more follow up questions, you must find the best venue options based on the following criteria:
    - Price (lowest)
    - Capacity (exact match)
    - Reviews (highest)
    You may need to make multiple searches to iteratively find the best options. 
    You have a suggested limit of 12 web searches. Count every web_search call you make.
    After 12 searches, you should stop searching and summarize the best options you have
    found so far.
    """
)

In [ ]:
# Playlist agent
playlist_agent = create_agent(
    model=model,
    tools=[query_playlist_db],
    system_prompt="""
    You are a playlist specialist. Query the sql database and curate the perfect playlist for a wedding given a genre.
    Understand the database schema first so that you can query it effectively.
    Once you have your playlist, calculate the total duration and cost of the playlist, each song has an associated price.
    If you run into errors when querying the database, try to fix them by making changes to the query.
    Do not come back empty handed, keep trying to query the db until you find a list of songs.

    This is a SQLite database. Before writing any data queries, first discover the schema.
    """
)

## Main Coordinator


In [9]:
from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langgraph.types import Command

@tool
async def search_flights(runtime: ToolRuntime) -> str:
    """Travel agent searches for flights to the desired destination wedding location."""
    origin = runtime.state["origin"]
    destination = runtime.state["destination"]
    response = await travel_agent.ainvoke({"messages": [HumanMessage(content=f"Find flights from {origin} to {destination}")]})
    return response['messages'][-1].content

@tool
def search_venues(runtime: ToolRuntime) -> str:
    """Venue agent chooses the best venue for the given location and capacity."""
    destination = runtime.state["destination"]
    capacity = runtime.state["guest_count"]
    query = f"Find wedding venues in {destination} for {capacity} guests"
    response = venue_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def suggest_playlist(runtime: ToolRuntime) -> str:
    """Playlist agent curates the perfect playlist for the given genre."""
    genre = runtime.state["genre"]
    query = f"Find {genre} tracks for wedding playlist"
    response = playlist_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def update_state(origin: str, destination: str, guest_count: str, genre: str, runtime: ToolRuntime) -> str:
    """Update the state when you know all of the values: origin, destination, guest_count, genre. 
    This tool must be called alone, without any other tool calls. It must complete and return to make,
    the information available to other tools."""
    return Command(update={
        "origin": origin, 
        "destination": destination, 
        "guest_count": guest_count, 
        "genre": genre, 
        "messages": [ToolMessage("Successfully updated state", tool_call_id=runtime.tool_call_id)]}
        )


In [10]:
from langchain.agents import create_agent

coordinator = create_agent(
    model=model,
    tools=[search_flights, search_venues, suggest_playlist, update_state],
    state_schema=WeddingState,
    system_prompt="""
    You are a wedding coordinator. 
    First find all the information you need to update the state. When you have the information, update the state.
    Once that has completed and returned, you can delegate the tasks 
    to your specialists for flights, venues, and playlists.
    Once you have received their answers, coordinate the perfect wedding for me.
    """
)


## Test


In [11]:
from langchain.messages import HumanMessage

response = await coordinator.ainvoke(
    {
        "messages": [HumanMessage(content="I'm from London and I'd like a wedding in Paris for 100 guests, jazz-genre")],
    },
    config={"tags": ["WP"], "recursion_limit": 40},  #tag traces to make them easy to find in Langsmith. Increase number of steps the agent can take to 40.
)

In [12]:
from pprint import pprint

pprint(response)

{'destination': 'Paris',
 'genre': 'jazz',
 'guest_count': '100',
 'messages': [HumanMessage(content="I'm from London and I'd like a wedding in Paris for 100 guests, jazz-genre", additional_kwargs={}, response_metadata={}, id='85dd9954-bada-4010-8cc3-0c7427945e69'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 344, 'total_tokens': 381, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gemma-4-E4B-it-UD-Q4_K_XL.gguf', 'system_fingerprint': 'b9670-02810c7aa', 'id': 'chatcmpl-ZAy2OStRjOP9YBFAlHmbWAOFpnsPKkxj', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a084a3-480f-7c01-b7de-5be2d0010fe1-0', tool_calls=[{'name': 'update_state', 'args': {'destination': 'Paris', 'genre': 'jazz', 'guest_count': '100', 'origin': 'London'}, 'id': 'rMfEMkHuJEzugF5y

In [39]:
print(response["messages"][-1].content)

This is such an exciting time! Planning a wedding in Paris with a jazz theme sounds incredibly romantic and chic.

I have gathered all the necessary information from my specialists:

### ✈️ Flights (London to Paris)
The flight agent needs a specific date to provide you with accurate quotes. I have made a preliminary suggestion to search for **May 15, 2024**, as this is often a beautiful and less crowded time for weddings in Paris. **Please let me know if this date works for you, or if you have a different date in mind!** Once you confirm, I can proceed with the flight search.

### 🏰 Venues (Paris for 100 Guests)
The venue specialist is ready to suggest beautiful options for 100 guests in Paris once we have a clearer picture of your budget and preferred style (e.g., historic château, intimate bistro, grand ballroom).

### 🎶 Playlist (Jazz Genre)
The playlist specialist has delivered a perfect curated soundtrack for your celebration!
*   **Total Tracks:** 20
*   **Total Estimated Duratio

# TESTING

In [31]:
"""
Test harness for wedding planning subagents:
- travel_agent (MCP / Kiwi flights)
- venue_agent (Tavily search)
- playlist_agent (SQLite Chinook database)

This script invokes each subagent directly with mock scenarios, tracks tool calls,
and tests both isolated and concurrent executions.
"""

import asyncio
import traceback
from pprint import pprint
from langchain.messages import HumanMessage


def print_agent_steps(agent_name: str, response: dict) -> None:
    """Helper to inspect the conversation and intermediate tool calls."""
    messages = response.get("messages", [])
    print(f"\n{'='*25} {agent_name} Execution Steps ({len(messages)} messages) {'='*25}")
    
    for idx, msg in enumerate(messages):
        msg_type = msg.__class__.__name__
        if msg_type == "HumanMessage":
            print(f"[{idx}] User: {msg.content}")
        elif msg_type == "AIMessage":
            if getattr(msg, "tool_calls", None):
                print(f"[{idx}] Agent invoked tools:")
                for tool_call in msg.tool_calls:
                    print(f"     -> {tool_call['name']}({tool_call['args']})")
            if msg.content:
                print(f"[{idx}] Agent thought/text: {msg.content[:200]}...")
        elif msg_type == "ToolMessage":
            preview = str(msg.content)[:150].replace("\n", " ")
            print(f"[{idx}] Tool Output ({getattr(msg, 'name', 'tool')}): {preview}...")
        else:
            print(f"[{idx}] {msg_type}: {str(msg.content)[:100]}...")

    print(f"\n--- Final Answer for {agent_name} ---")
    if messages:
        print(messages[-1].content)
    else:
        print("[No messages returned]")
    print(f"{'='*70}\n")


async def run_agent_safely(agent, input_dict: dict, agent_name: str) -> dict | None:
    """Dispatches to ainvoke if available, otherwise falls back to invoke in a worker thread."""
    try:
        if hasattr(agent, "ainvoke"):
            return await agent.ainvoke(input_dict)
        else:
            # Run blocking synchronous invoke in thread pool to avoid blocking the event loop
            return await asyncio.to_thread(agent.invoke, input_dict)
    except Exception as e:
        print(f"❌ {agent_name} failed with exception: {type(e).__name__}: {e}")
        traceback.print_exc()
        return None


async def test_travel_agent(origin: str = "London", destination: str = "Paris"):
    print(f"\n🛫 [TEST] Starting Travel Agent Test ({origin} -> {destination})...")
    if "travel_agent" not in globals():
        print("❌ 'travel_agent' not found in globals. Run setup cells first.")
        return None

    prompt = f"Find one-way economy flights from {origin} to {destination} for 1 adult."
    response = await run_agent_safely(
        travel_agent,
        {"messages": [HumanMessage(content=prompt)]},
        "Travel Agent"
    )

    if response:
        print("✅ Travel Agent invocation completed successfully.")
        print_agent_steps("Travel Agent", response)
    return response


async def test_venue_agent(destination: str = "Paris", guests: str = "100"):
    print(f"\n🏰 [TEST] Starting Venue Agent Test ({destination}, {guests} guests)...")
    if "venue_agent" not in globals():
        print("❌ 'venue_agent' not found in globals. Run setup cells first.")
        return None

    prompt = f"Find wedding venues in {destination} for {guests} guests. Adhere to your search limits and rules."
    response = await run_agent_safely(
        venue_agent,
        {"messages": [HumanMessage(content=prompt)]},
        "Venue Agent"
    )

    if response:
        print("✅ Venue Agent invocation completed successfully.")
        print_agent_steps("Venue Agent", response)
    return response


async def test_playlist_agent(genre: str = "Jazz"):
    print(f"\n🎷 [TEST] Starting Playlist Agent Test (Genre: {genre})...")
    if "playlist_agent" not in globals():
        print("❌ 'playlist_agent' not found in globals. Run setup cells first.")
        return None

    prompt = f"Find {genre} tracks for a wedding playlist from the database. Calculate total duration and price."
    response = await run_agent_safely(
        playlist_agent,
        {"messages": [HumanMessage(content=prompt)]},
        "Playlist Agent"
    )

    if response:
        print("✅ Playlist Agent invocation completed successfully.")
        print_agent_steps("Playlist Agent", response)
    return response


# =====================================================================
# Execution Options:
# 1. Individual Sequential Test (recommended first to isolate errors)
# 2. Concurrent Execution via asyncio.gather
# =====================================================================

# --- OPTION 1: Run individually ---
await test_travel_agent("London", "Paris")
# await test_venue_agent("Paris", "100")
# await test_playlist_agent("Jazz")

# --- OPTION 2: Run all three in parallel ---
# results = await asyncio.gather(
#     test_travel_agent("London", "Paris"),
#     test_venue_agent("Paris", "100"),
#     test_playlist_agent("Jazz"),
#     return_exceptions=True
# )


🛫 [TEST] Starting Travel Agent Test (London -> Paris)...
✅ Travel Agent invocation completed successfully.

========================= Travel Agent Execution Steps (6 messages) =========================
[0] User: Find one-way economy flights from London to Paris for 1 adult.
[1] Agent invoked tools:
     -> get_current_date({})
[2] Tool Output (get_current_date): 2026-09-09...
[3] Agent invoked tools:
     -> search-flight({'adults': 1, 'cabinClass': 'M', 'departureDate': '09/10/2026', 'flyFrom': 'London', 'flyTo': 'Paris'})
[4] Tool Output (search-flight): [{'type': 'text', 'text': '{\n  "query": "London → Paris on 09/10/2026, 1 adult, economy",\n  "currency": "EUR",\n  "passengers": {\n    "adults": 1,\...
[5] Agent thought/text: I have found several one-way economy flights from London to Paris for you. Since you did not specify a date, I have selected **October 9, 2026**, as the best future date within the next year. This dat...

--- Final Answer for Travel Agent ---
I have found se

{'messages': [HumanMessage(content='Find one-way economy flights from London to Paris for 1 adult.', additional_kwargs={}, response_metadata={}, id='86a442e1-3117-4429-8274-2aab363b036e'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 2671, 'total_tokens': 2682, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gemma-4-E4B-it-UD-Q4_K_XL.gguf', 'system_fingerprint': 'b9670-02810c7aa', 'id': 'chatcmpl-qUy5lyhs0vkpz268Tk4jRKhxsMYOW1Un', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a084c1-4637-7f80-965c-ebab6adce970-0', tool_calls=[{'name': 'get_current_date', 'args': {}, 'id': 'VtxEZG9HBicoXDcPtnqreSxT8lQA3xfz', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 2671, 'output_tokens': 11, 'total_tokens': 2682, 'input_token_detail

link to trace: https://smith.langchain.com/public/7b5fe668-d3e3-4af4-b513-a8cacc0c9e84/r